# Character-level language model

A GPT-style character-level language model in the Karpathy char-rnn/minGPT
tradition: sliding window over a corpus, next-character prediction at every
position.

It reuses the same decomposed transformer as the sorting example, configured
for language modeling: a larger vocabulary (65 characters) and a longer
context window (64).

**CLI equivalents:** `make example-gpt` (embedded one-sentence corpus, 30
epochs) and `make example-gpt-full` (tiny-shakespeare, 1000 epochs)


## Architecture

Same transformer stack as the sorting task, configured for language modeling:
- Vocabulary: 65 characters
- Sequence length: 64 (the sliding context window)
- Embedding dim 64, 4 heads of dim 16, 2 pre-norm blocks

Input: one `[64]` window of token indices. Output: `[64, 65]` per-position
logits. Loss: cross-entropy on all positions (standard LM loss).


In [1]:
:browse Ml.Nn.Transformer

MkTransformerBlock : (1 _ : Attention dModel numHeads headDim ex dt g) -> (1 _ : LayerNorm dModel dModel ex dt g) -> (1 _ : LayerNorm dModel dModel ex dt g) -> TMat (4 * dModel) dModel ex dt g -> TMat dModel (4 * dModel) ex dt g -> TransformerBlock numHeads headDim dModel dModel ex dt g
TransformerBlock : Nat -> Nat -> Nat -> Nat -> (0 _ : Executor) -> (0 _ : DType) -> (0 _ : GradMode) -> Type
transformerBlock : KnownGrad g => Backend ex dt => Init (TransformerBlock numHeads headDim dModel dModel ex dt g)


## Model construction

The model record mirrors the sorting transformer's — an `Embedding 65 64`, a
linear `Seq` body of `TransformerBlock`s plus a final `LayerNorm`, a bias-free
output head, and a sinusoidal positional encoding. Corpus tokenization and the
sliding-window batch generator live in the compiled example
(`packages/idris-ml-examples/src/Example/Gpt.idr`).


## Training

AdamW (betas (0.9, 0.99), weight decay 0.1, grad-clip norm 1.0) under a
cosine-with-warmup schedule, which `fit` ticks once per epoch. From
`Example/Gpt.idr`:

```idris
opt0 <- adamW {ex=ExampleExecutor} cfg.lr 0.1 ({ beta2 := 0.99, clip := NormClip 1.0 } defaultOpts)
let opt = withSchedule schedule opt0

(MkBang (epochsDone, _) # trained) <-
  fitSupervised {ex=ExampleExecutor} opt batchLossL
                 (generate (gptBatch trainIndices trainLen BatchSize)) trainCfg model
```

Run the smoke config: `make example-gpt GPT_ARGS="--epochs 30"`.


## Autoregressive generation

After training, the model generates text by repeatedly:
1. Forward pass on the context window
2. Sample or argmax the last position's logits
3. Append the new token, shift the window

The final report prints bits-per-character on a held-out corpus slice and a
generated sample.


## Scaling up

The embedded corpus is one sentence (67 characters) — a smoke-test config that
demonstrates the pipeline end-to-end, not a language model. For a real run,
train on tiny-shakespeare (downloaded by `make`):

```bash
make example-gpt-full                          # 1000 epochs
make example-gpt-full GPT_ARGS="--epochs 200"  # shorter run
```


## PyTorch comparison

```python
# Karpathy's minGPT pattern
model = GPT(vocab_size=65, block_size=64, n_embd=64,
            n_head=4, n_layer=2)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(1000):
    x, y = get_batch(corpus, block_size=64)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, 65), y.view(-1))
    loss.backward()
    optimizer.step()
```

See `pytorch/torch_ref/scripts/gpt.py` for the full reference.


Next: [NTM](ntm.ipynb) — neural networks with external memory.
